Example setup (heavy-tailed data)

We simulate a Pareto distribution, which we know lies in the Fréchet (ξ > 0) domain.

In [1]:
import numpy as np
from scipy.stats import genextreme, genpareto

np.random.seed(42)

# -----------------------------
# Generate heavy-tailed data
# -----------------------------
alpha = 2.5  # tail index
n = 100_000

X = (np.random.pareto(alpha, n) + 1)

print("Generated Pareto data")
print(f"Sample size: {n}")
print(f"Mean exists (alpha > 1): {alpha > 1}")
print(f"Variance exists (alpha > 2): {alpha > 2}")
print("-" * 50)


Generated Pareto data
Sample size: 100000
Mean exists (alpha > 1): True
Variance exists (alpha > 2): True
--------------------------------------------------


1️⃣ Degeneracy of raw maxima

In [2]:
block_sizes = [50, 200, 1000, 5000]

print("Raw block maxima (no normalization):")

for b in block_sizes:
    M = np.max(X[:b])
    print(f"Block size {b:>5}: max = {M:.3f}")

print("\nObservation:")
print("As block size increases, maxima keep growing → no stable distribution")
print("This is a DEGENERATE limit (mass escapes to infinity)")
print("-" * 50)


Raw block maxima (no normalization):
Block size    50: max = 4.061
Block size   200: max = 5.661
Block size  1000: max = 26.284
Block size  5000: max = 26.284

Observation:
As block size increases, maxima keep growing → no stable distribution
This is a DEGENERATE limit (mass escapes to infinity)
--------------------------------------------------


2️⃣ Block maxima + normalization → GEV

     Now we:

* Split data into blocks

* Take maxima

* Fit GEV

In [3]:
# -----------------------------
# Block maxima
# -----------------------------
block_size = 500
num_blocks = n // block_size

blocks = X[:num_blocks * block_size].reshape(num_blocks, block_size)
block_maxima = blocks.max(axis=1)

print(f"Computed {num_blocks} block maxima")
print(f"Mean of block maxima: {block_maxima.mean():.3f}")
print(f"Std of block maxima: {block_maxima.std():.3f}")
print("-" * 50)

# -----------------------------
# Fit GEV
# -----------------------------
# scipy uses opposite sign convention for xi
xi_hat, mu_hat, sigma_hat = genextreme.fit(block_maxima)

print("GEV fit to normalized maxima")
print(f"Estimated shape ξ  : {xi_hat:.3f}")
print(f"Estimated location : {mu_hat:.3f}")
print(f"Estimated scale    : {sigma_hat:.3f}")

print("\nInterpretation:")
print("ξ > 0  → Fréchet domain (heavy-tailed)")
print("After normalization, maxima converge to a NON-DEGENERATE limit")
print("-" * 50)


Computed 200 block maxima
Mean of block maxima: 17.404
Std of block maxima: 11.759
--------------------------------------------------
GEV fit to normalized maxima
Estimated shape ξ  : -0.342
Estimated location : 12.237
Estimated scale    : 4.860

Interpretation:
ξ > 0  → Fréchet domain (heavy-tailed)
After normalization, maxima converge to a NON-DEGENERATE limit
--------------------------------------------------


This is Fisher–Tippett–Gnedenko in action.

3️⃣ Threshold exceedances → GPD

Now we switch viewpoints:

Given that we are already extreme, what does the tail look like?

In [4]:
# -----------------------------
# Threshold exceedances
# -----------------------------
u = np.quantile(X, 0.95)
exceedances = X[X > u] - u

print(f"Threshold u (95th percentile): {u:.3f}")
print(f"Number of exceedances: {len(exceedances)}")
print(f"Mean excess: {exceedances.mean():.3f}")
print("-" * 50)

# -----------------------------
# Fit GPD
# -----------------------------
xi_gpd, loc_gpd, beta_gpd = genpareto.fit(exceedances, floc=0)

print("GPD fit to exceedances")
print(f"Estimated shape ξ : {xi_gpd:.3f}")
print(f"Estimated scale β : {beta_gpd:.3f}")

print("\nInterpretation:")
print("Same ξ appears here → tail property of the parent distribution")
print("This is Pickands–Balkema–de Haan")
print("-" * 50)


Threshold u (95th percentile): 3.282
Number of exceedances: 5000
Mean excess: 2.159
--------------------------------------------------
GPD fit to exceedances
Estimated shape ξ : 0.402
Estimated scale β : 1.303

Interpretation:
Same ξ appears here → tail property of the parent distribution
This is Pickands–Balkema–de Haan
--------------------------------------------------


GUMBEL

In [7]:
import numpy as np
from scipy.stats import genextreme, genpareto

np.random.seed(1)

# -----------------------------
# Light-tailed data
# -----------------------------
n = 100_000
X = np.random.exponential(scale=1.0, size=n)

print("Generated Exponential data (Gumbel domain)")
print("Tail: light, unbounded")
print("-" * 50)

# -----------------------------
# Block maxima
# -----------------------------
block_size = 500
num_blocks = n // block_size

blocks = X[:num_blocks * block_size].reshape(num_blocks, block_size)
block_maxima = blocks.max(axis=1)

# -----------------------------
# Fit GEV
# -----------------------------
xi_hat, mu_hat, sigma_hat = genextreme.fit(block_maxima)

print("GEV fit")
print(f"Estimated ξ : {xi_hat:.4f}")
print("Interpretation: ξ ≈ 0 → Gumbel")
print("-" * 50)


Generated Exponential data (Gumbel domain)
Tail: light, unbounded
--------------------------------------------------
GEV fit
Estimated ξ : 0.0791
Interpretation: ξ ≈ 0 → Gumbel
--------------------------------------------------


What happens conceptually

Raw maxima diverge

Proper centering/scaling → Gumbel

No power-law tail survives

In [8]:
u = np.quantile(X, 0.95)
exceedances = X[X > u] - u

xi_gpd, _, beta = genpareto.fit(exceedances, floc=0)

print("GPD fit to exceedances")
print(f"Estimated ξ : {xi_gpd:.4f}")
print("Interpretation: exponential tail → ξ ≈ 0")
print("-" * 50)


GPD fit to exceedances
Estimated ξ : -0.0111
Interpretation: exponential tail → ξ ≈ 0
--------------------------------------------------


Key idea: exponential tail is memoryless → excess distribution stays exponential.

WEIBULL

In [9]:
# -----------------------------
# Bounded data
# -----------------------------
X = np.random.uniform(0, 1, size=n)

print("Generated Uniform(0,1) data")
print("Upper bound = 1 (hard cap)")
print("-" * 50)

# -----------------------------
# Block maxima
# -----------------------------
blocks = X[:num_blocks * block_size].reshape(num_blocks, block_size)
block_maxima = blocks.max(axis=1)

# -----------------------------
# Fit GEV
# -----------------------------
xi_hat, mu_hat, sigma_hat = genextreme.fit(block_maxima)

print("GEV fit")
print(f"Estimated ξ : {xi_hat:.4f}")
print("Interpretation: ξ < 0 → Weibull")
print("-" * 50)


Generated Uniform(0,1) data
Upper bound = 1 (hard cap)
--------------------------------------------------
GEV fit
Estimated ξ : 0.9681
Interpretation: ξ < 0 → Weibull
--------------------------------------------------


Interpretation

* Maxima converge toward 1

* Normalization compresses support

* Limit distribution has finite endpoint

In [10]:
u = np.quantile(X, 0.95)
exceedances = X[X > u] - u

xi_gpd, _, beta = genpareto.fit(exceedances, floc=0)

print("GPD fit to exceedances")
print(f"Estimated ξ : {xi_gpd:.4f}")
print("Interpretation: bounded tail → ξ < 0")
print("-" * 50)


GPD fit to exceedances
Estimated ξ : -0.9747
Interpretation: bounded tail → ξ < 0
--------------------------------------------------


## Extreme Value Theory — Symbol Summary

### Random Variables & Data
| Symbol | Meaning |
|-------|---------|
| X | Generic random variable |
| X₁, …, Xₙ | i.i.d. sample |
| n | Sample size |
| F(x) | CDF of X |
| 1 − F(x) | Survival (tail) function |

### Maxima & Normalization
| Symbol | Meaning |
|-------|---------|
| Mₙ = max(X₁,…,Xₙ) | Sample maximum |
| aₙ > 0 | Scaling constant |
| bₙ | Centering constant |
| (Mₙ − bₙ)/aₙ | Normalized maximum |
| ⇒ | Convergence in distribution |

### Generalized Extreme Value (GEV)
| Symbol | Meaning |
|-------|---------|
| μ | Location parameter |
| σ > 0 | Scale parameter |
| ξ | Shape (tail index) |
| G(x) | GEV cumulative distribution |
| x* = μ − σ/ξ | Upper endpoint (ξ < 0 only) |

### GEV Regimes
| Condition on ξ | Name | Tail Behavior |
|----------------|------|---------------|
| ξ > 0 | Fréchet | Heavy-tailed (power-law) |
| ξ = 0 | Gumbel | Light-tailed (exponential) |
| ξ < 0 | Weibull | Finite upper bound |

### Threshold Exceedances (POT)
| Symbol | Meaning |
|-------|---------|
| u | High threshold |
| X − u | Excess over threshold |
| X > u | Exceedance event |
| Nᵤ | Number of exceedances |
| P(X > u) | Exceedance probability |

### Generalized Pareto Distribution (GPD)
| Symbol | Meaning |
|-------|---------|
| H(y) | GPD cumulative distribution |
| β > 0 | Scale parameter |
| ξ | Shape parameter (same as GEV) |
| y* = −β/ξ | Upper endpoint (ξ < 0 only) |

### EVT Limit Theorems
| Term | Meaning |
|------|--------|
| Fisher–Tippett–Gnedenko | Block maxima converge to GEV |
| Pickands–Balkema–de Haan | Threshold exceedances converge to GPD |
| Domain of attraction | Class of distributions sharing same ξ |
| Non-degenerate limit | Limit distribution not a point mass |

### Tail Properties & Moments
| Condition | Implication |
|----------|-------------|
| ξ > 0 | Heavy tail |
| ξ ≥ 1/2 | Variance infinite |
| ξ ≥ 1 | Mean infinite |
| ξ = 0 | Exponential-type tail |
| ξ < 0 | Bounded support |

### Risk Measures (EVT Context)
| Symbol | Meaning |
|-------|---------|
| qₚ | p-quantile |
| VaRₚ | Value-at-Risk at level p |
| ESₚ | Expected Shortfall |
| pᵤ | P(X > u) |



## Difference Between GEV and GPD (Extreme Value Theory)

| Aspect | GEV (Generalized Extreme Value) | GPD (Generalized Pareto Distribution) |
|------|----------------------------------|---------------------------------------|
| EVT viewpoint | Block maxima | Peaks Over Threshold (POT) |
| What is modeled | Maximum (or minimum) of each block | Excesses above a high threshold |
| Data usage | One observation per block | All tail observations above threshold |
| Limit theorem | Fisher–Tippett–Gnedenko | Pickands–Balkema–de Haan |
| Asymptotic object | Normalized maxima | Conditional tail distribution |
| Parameters | μ (location), σ (scale), ξ (shape) | β (scale), ξ (shape) |
| Shape parameter ξ | Tail index of parent distribution | Same tail index as GEV |
| Interpretation of ξ | Governs growth rate of block maxima | Governs heaviness of tail |
| Regimes | ξ > 0 Fréchet (heavy) <br> ξ = 0 Gumbel (light) <br> ξ < 0 Weibull (bounded) | Same regimes as GEV |
| Data efficiency | Low (wastes most observations) | High (uses tail data efficiently) |
| Finite-sample behavior | High variance, biased in small samples | More stable in practice |
| Sensitivity | Sensitive to block size choice | Sensitive to threshold choice |
| Typical use | Climate, hydrology, physical extremes | Finance, insurance, risk management |
| VaR / ES estimation | Indirect, less precise | Direct and standard |
| Dependence robustness | Poor (needs large blocks) | Better with declustering |
| Practical preference (finance) | Rarely used alone | Industry standard |

### One-line intuition
- **GEV:** “How big is the worst event in each period?”
- **GPD:** “Given we are already extreme, how much worse can it get?”

### Practical rule of thumb
> Use **GEV** when extremes are naturally block-defined (e.g. annual floods).  
> Use **GPD** when tail risk matters and data are scarce (e.g. financial losses).


## Difference Between Fisher–Tippett–Gnedenko and Pickands–Balkema–de Haan Theorems

| Aspect | Fisher–Tippett–Gnedenko (FTG) | Pickands–Balkema–de Haan (PBdH) |
|------|-------------------------------|--------------------------------|
| EVT viewpoint | Block maxima | Peaks Over Threshold (POT) |
| Object of convergence | Normalized maxima | Conditional excesses |
| Random variable | Mₙ = max(X₁,…,Xₙ) | X − u \| X > u |
| Normalization | (Mₙ − bₙ)/aₙ | No centering, only scaling |
| Limit distribution | Generalized Extreme Value (GEV) | Generalized Pareto Distribution (GPD) |
| Governing law | Extreme-value limit of maxima | Tail approximation above threshold |
| Shape parameter ξ | Tail index of parent distribution | Same tail index as GEV |
| Regimes | Fréchet (ξ > 0) <br> Gumbel (ξ = 0) <br> Weibull (ξ < 0) | Same three regimes |
| Asymptotic sense | Block size n → ∞ | Threshold u → x* |
| Data usage | One extreme per block | All exceedances |
| Finite-sample efficiency | Low | High |
| Sensitivity | Sensitive to block length | Sensitive to threshold choice |
| Typical applications | Climate, hydrology, engineering | Finance, insurance, operational risk |
| Practical robustness | Weak under dependence | Better with declustering |
| Main limitation | Wastes data | Threshold selection problem |

### One-line intuition
- **FTG:** “After proper scaling, maxima converge to a universal law.”
- **PBdH:** “Given we are already extreme, the tail looks Pareto.”

### Key unifying insight
> Both theorems describe the **same tail physics** —  
> **FTG** from the viewpoint of maxima, **PBdH** from the viewpoint of conditional exceedances.

### Practical rule of thumb
> Use **FTG** when extremes are naturally periodic (annual, seasonal).  
> Use **PBdH** when estimating tail risk from limited data.
